In [1]:
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
import json
from dotenv import load_dotenv

d:\anaconda3\envs\tf2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@tool
def get_weather(location: str) -> str:
    '''
    获取指定地点的天气信息
    
    Args:
        location: 地点名称 e.g. 北京, 上海
    
    Returns:
        该地点的天气描述
    '''
    weather_data = {
        "北京": "晴朗，气温25°C，空气质量优",
        "上海": "多云，气温28°C，适合户外活动",
        "广州": "小雨，气温30°C，记得带伞",
        "深圳": "雷阵雨，气温29°C，注意安全",
        "成都": "阴天，气温22°C，凉爽舒适",
        "杭州": "晴天，气温26°C，西湖美景"
    }
    if location in weather_data:
        return f'{location}的天气: {weather_data[location]}'
    else:
        return f'{location}的天气: 晴朗'


In [20]:
load_dotenv(r'D:\github\练习\key.env')
model = ChatZhipuAI(
    model='glm-4.5-flash',
    temperature=0.1
)

model_with_tools = model.bind_tools([get_weather])
tools = {'get_weather': get_weather}


In [21]:
messages = []
template = ChatPromptTemplate([
    ('system', '你是一个善于调用工具的大模型, 根据用户的问题判断是否需要调用已有的工具, 结合工具返回的信息回答用户的问题'),
    ('user', '{question}')
])
message = template.format_messages(question='北京的天气如何')
ai_message = model_with_tools.invoke(message)
messages.append(ai_message)


In [22]:
ai_message.tool_calls

[{'name': 'get_weather',
  'args': {'location': '北京'},
  'id': 'call_-7806723700434136577',
  'type': 'tool_call'}]

In [ ]:
if ai_message.tool_calls:
    for tool in ai_message.tool_calls:
        selected_tool = tools[tool['name']]
        tool_message = selected_tool.invoke(tool)
        messages.append(tool_message)


In [26]:
model_with_tools.invoke(messages)

HTTPStatusError: Client error '400 Bad Request' for url 'https://open.bigmodel.cn/api/paas/v4/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400

In [25]:
messages

[AIMessage(content='\n我来帮您查询北京的天气情况。\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"location":"北京"}', 'name': 'get_weather'}, 'id': 'call_-7806723700434136577', 'index': 0, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_tokens': 216, 'prompt_tokens_details': {'cached_tokens': 215}, 'total_tokens': 282}, 'model_name': 'glm-4.5-flash', 'finish_reason': 'tool_calls'}, id='run--019cf5b3-cb44-7ea2-8173-8bd8083c1076-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '北京'}, 'id': 'call_-7806723700434136577', 'type': 'tool_call'}]),
 ToolMessage(content='北京的天气: 晴朗，气温25°C，空气质量优', name='get_weather', tool_call_id='call_-7806723700434136577')]